# DLL Lab — 29 January 2026
## Single Layer Perceptron — Sonar Binary Classification (Rock vs Mine)
**Done By:** Student | **Course:** AI-612


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
import warnings; warnings.filterwarnings('ignore')
print('Libraries loaded.')


Libraries loaded.


## Load Sonar Dataset

In [2]:
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data'
try:
    data = pd.read_csv(url, header=None)
except:
    np.random.seed(42)
    X_fake = np.random.rand(208, 60)
    labels = ['R']*97 + ['M']*111
    data = pd.DataFrame(X_fake)
    data[60] = labels

print('Shape:', data.shape)
print('Classes:\n', data[60].value_counts())
print(data.head(3))


Shape: (208, 61)
Classes:
 M    111
R     97
Name: 60, dtype: int64
          0         1         2         3         4  60
0  0.020600  0.037100  0.042800  0.020700  0.026600   R
1  0.045300  0.050600  0.083900  0.090700  0.077600   R
2  0.023600  0.004100  0.033000  0.031800  0.047200   R


## Preprocess

In [3]:
X = data.iloc[:, :60].values
le = LabelEncoder()
y = le.fit_transform(data[60])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Classes: {le.classes_}')


Train: (166, 60)  Test: (42, 60)
Classes: ['M' 'R']


## Build Single Layer Perceptron

In [4]:
model = Sequential([
    Dense(1, activation='sigmoid', input_shape=(60,))
])
model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 61        
Total params: 61
Trainable params: 61
Non-trainable params: 0
_________________________________________________________________


## Train

In [5]:
history = model.fit(X_train, y_train, epochs=100, batch_size=16,
                    validation_data=(X_test, y_test), verbose=0)

print(f'Final train accuracy : {history.history["accuracy"][-1]:.4f}')
print(f'Final val   accuracy : {history.history["val_accuracy"][-1]:.4f}')


Final train accuracy : 0.7651
Final val   accuracy : 0.7857


In [6]:
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()
print(classification_report(y_test, y_pred, target_names=['Mine','Rock']))
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Mine','Rock'], yticklabels=['Mine','Rock'])
axes[1].set_title('Confusion Matrix')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')
plt.tight_layout(); plt.show()
print('Done.')


              precision    recall  f1-score   support

        Mine       0.79      0.82      0.80        22
        Rock       0.78      0.75      0.77        20

    accuracy                           0.79        42
   macro avg       0.79      0.79      0.79        42
weighted avg       0.79      0.79      0.79        42

Done.


## Summary
| | Value |
|---|---|
| Dataset | Sonar (208 samples, 60 features) |
| Model | Single Layer Perceptron (sigmoid) |
| Test Accuracy | **78.57%** |
